# 02 · Simulated data and the forecasting dataset

**Purpose.** Show how two years of realistic data were produced, and how they become the forecasting table.

```
 2025: 12 month runs (train; last 5 days/month = calibration)   2026: 12 month runs (test, untouched)
 each run: 1 Hz packets + hourly-issued 48 h weather forecasts ─► clean ─► hourly actuals ─► feature table
```
**Key takeaways**
1. The weather forecasts carry realistic errors that grow with lead time; they were not tuned.
2. Features use only information available at the issue time (tested).
3. Spring/summer curtailment removes ~10 % of PV energy.

In [1]:
%matplotlib inline
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from aerosolar.config import Config, REPO_ROOT
from aerosolar.sim.realism import _style, S1, S2, S3, INK, INK2
_style(); plt.rcParams['figure.dpi'] = 110
cfg = Config.load()
DATA = REPO_ROOT / 'data'
pd.set_option('display.width', 140); pd.set_option('display.max_columns', 30)

In [2]:
from aerosolar.ml import campaign as C
pd.DataFrame(C.runs())[['split', 'year', 'month', 'seed', 'hours']].pivot(index='month', columns='split', values='seed')

split,test,train
month,,
1,201,101
2,202,102
3,203,103
4,204,104
5,205,105
6,206,106
7,207,107
8,208,108
9,209,109


## 1. Weather-forecast error vs lead time
The forecast the model receives, compared with what actually happened (one test month):

In [3]:
run = C.run_dir([r for r in C.runs() if r['split'] == 'test'][5])
fc = pd.read_parquet(run / 'forecast_issued.parquet'); tr = pd.read_parquet(run / 'weather_hourly_truth.parquet').set_index('valid_time')
fc['wind_err'] = fc['wind_speed_10m'] - tr['wind_10m_equiv'].reindex(fc['valid_time']).to_numpy()
fc['ghi_err'] = fc['shortwave_radiation'] - tr['ghi'].reindex(fc['valid_time']).to_numpy()
g = fc.groupby('lead_h')
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].plot(g['wind_err'].std(), color=S1); ax[0].set_ylabel('wind error σ [m/s]'); ax[0].set_xlabel('lead [h]')
day = fc[tr['ghi'].reindex(fc['valid_time']).to_numpy() > 100]
ax[1].plot(day.groupby('lead_h')['ghi_err'].std(), color=S2); ax[1].set_ylabel('irradiance error σ [W/m²]'); ax[1].set_xlabel('lead [h]')
plt.show()

**What to notice.** Errors grow with lead time, as real NWP errors do. These set the ceiling for any forecaster.

## 2. Curtailment

In [4]:
rows = []
for r in C.runs():
    if r['split'] != 'train': continue
    h = pd.read_parquet(C.run_dir(r) / 'derived' / 'hourly.parquet')
    day = h['ghi_clear'] > 50
    rows.append({'month': r['month'], 'daylight hours with PV curtailed %': 100 * h.loc[day, 'float_frac'].mean(),
                 'mean battery SOC %': h['soc'].mean()})
pd.DataFrame(rows).set_index('month').round(1)

,daylight hours with PV curtailed %,mean battery SOC %
month,,
1,0.0,39.3
2,0.3,61.2
3,4.4,61.7
4,13.9,75.3
5,12.2,75.1
6,16.4,76.0
7,14.2,74.8
8,8.7,72.6
9,2.4,67.1


**What to notice.** In spring and summer the battery fills by early afternoon and the controller throttles the panel; in those months this costs 10–11 % of the available PV energy (measured on the simulator's internal record).

> **Assumption.** 5.7 W self-consumption and a 12 Ah battery; both decide how often this happens (CALIBRATION.md step 8).

## 3. The feature table

In [5]:
from aerosolar.ml.forecast_features import FEATURES
from aerosolar.ml.train import load_split
tab, _ = load_split('test')
print(len(tab), 'rows ·', len(FEATURES), 'features')
tab[['issue_time', 'valid_time', 'lead_h', 'sw', 'ws10', 'kt_fc', 'obs_soc', 'pv_yield_72h', 'sw_cum', 'y_pv', 'y_wind']].head(8)

420480 rows · 28 features


,issue_time,valid_time,lead_h,sw,ws10,kt_fc,obs_soc,pv_yield_72h,sw_cum,y_pv,y_wind
0,2025-12-31 20:00:00+00:00,2025-12-31 21:00:00+00:00,1,0.000000,0.116539,0.000000,NaN,NaN,0.000000,0.000000,0.0
1,2025-12-31 20:00:00+00:00,2025-12-31 22:00:00+00:00,2,0.000000,0.778427,0.000000,NaN,NaN,0.000000,0.000000,0.0
2,2025-12-31 20:00:00+00:00,2025-12-31 23:00:00+00:00,3,0.000000,1.128965,0.000000,NaN,NaN,0.000000,0.000000,0.0
3,2025-12-31 20:00:00+00:00,2026-01-01 00:00:00+00:00,4,0.000000,0.558190,0.000000,NaN,NaN,0.000000,0.000000,0.0
4,2025-12-31 20:00:00+00:00,2026-01-01 01:00:00+00:00,5,0.000000,1.252113,0.000000,NaN,NaN,0.000000,0.000000,0.0
5,2025-12-31 20:00:00+00:00,2026-01-01 02:00:00+00:00,6,0.000000,0.443166,0.000000,NaN,NaN,0.000000,0.000000,0.0
6,2025-12-31 20:00:00+00:00,2026-01-01 03:00:00+00:00,7,49.162229,0.080829,0.936694,NaN,NaN,49.162229,2.476861,0.0
7,2025-12-31 20:00:00+00:00,2026-01-01 04:00:00+00:00,8,227.244787,0.014204,0.919200,NaN,NaN,276.407017,9.546448,0.0


**What to notice.** One row per (issue time, valid hour). Observation features (`obs_*`, `pv_yield_72h`, persistence) only use
hours that ended by the issue time; `tests/test_forecast.py` corrupts all later data and checks the features do not change.
That test caught a real leak during development (48 h persistence).

## What we learned
* A fair forecasting dataset needs realistic forecast errors and strict time ordering.

## What changes with real data
Archive the real Open-Meteo forecasts every hour from day one: they become the training input.

## References
* Romano, Patterson, Candès (2019). *Conformalized quantile regression.* NeurIPS.
* Ke et al. (2017). *LightGBM: a highly efficient gradient boosting decision tree.* NeurIPS.
* Yang et al. (2020). *Verification of deterministic solar forecasts.* Solar Energy 210 (clear-sky index, smart persistence, skill scores).
* Hong et al. (2016). *Probabilistic energy forecasting: GEFCom2014 and beyond.* IJF 32.
* Haurwitz (1945). *Insolation in relation to cloudiness and cloud density.* J. Meteorology.
* Kamoji, Kedare, Prabhu (2009). *Performance tests on helical Savonius rotors.* Renewable Energy 34.